# 02 — Label Validation

Compute all labels, check prevalence, and validate with manual sampling.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd, yaml
from src.data.ingest import get_events, get_360_frames
from src.data.parse_events import parse_events
from src.data.parse_360 import parse_360_frames, get_frame_summary
from src.data.join_pass_frames import build_pass_instances
from src.labels.line_break import compute_line_break_labels
from src.labels.dangerous_progression import compute_downstream_labels
from src.labels.downstream_outcomes import compute_threat_gain
from src.labels.validation_sampling import (
    label_prevalence_table, label_sanity_checks,
    label_by_zone, label_by_pass_type, sample_positives, sample_negatives
)

with open("../configs/labels.yaml") as f:
    label_cfg = yaml.safe_load(f)


## 1. Load data (replace with full dataset after ingestion)

In [2]:

pass_instances = pd.read_parquet("../data/processed/pass_instances.parquet")
frames = pd.read_parquet("../data/processed/frames_360.parquet")
events_df = pd.read_parquet("../data/processed/events_parsed.parquet")
print(f"Pass instances: {len(pass_instances):,}")
print(f"360 frame rows: {len(frames):,}")
print(f"Parsed events:  {len(events_df):,}")


Pass instances: 50,688
360 frame rows: 3,084,876
Parsed events:  373,682


## 2. Compute labels

In [3]:

# Labels already computed during processing — verify they exist
label_cols = ["strict_line_break", "loose_line_break",
              "dangerous_progression_k", "final_third_entry_k",
              "box_entry_k", "shot_within_k", "threat_gain"]
present = [c for c in label_cols if c in pass_instances.columns]
missing = [c for c in label_cols if c not in pass_instances.columns]
print(f"Label columns present: {present}")
if missing:
    print(f"Missing: {missing}")
else:
    print("All label columns present ✓")


Label columns present: ['strict_line_break', 'loose_line_break', 'dangerous_progression_k', 'final_third_entry_k', 'box_entry_k', 'shot_within_k', 'threat_gain']
All label columns present ✓


## 3. Label prevalence

In [4]:

label_cols = ["strict_line_break", "loose_line_break",
              "dangerous_progression_k", "final_third_entry_k",
              "box_entry_k", "shot_within_k"]
prevalence = label_prevalence_table(pass_instances, label_cols)
prevalence


,label,n_total,n_known,n_positive,n_negative,n_nan,prevalence
0,strict_line_break,50688,26515,9619,16896,24173,0.362776
1,loose_line_break,50688,26515,11471,15044,24173,0.432623
2,dangerous_progression_k,50688,50688,21928,28760,0,0.432607
3,final_third_entry_k,50688,50688,21922,28766,0,0.432489
4,box_entry_k,50688,50688,5104,45584,0,0.100694
5,shot_within_k,50688,50688,899,49789,0,0.017736


## 4. Sanity checks

In [5]:

results = label_sanity_checks(pass_instances)
for check, passed in results.items():
    status = "✓" if passed else "✗"
    print(f"{status}  {check}")


✓  strict_le_loose
✓  final_third_entry_le_dp
✓  box_entry_le_dp
✓  shot_within_le_dp
✓  dp_or_condition
✓  threat_gain_range
✓  no_nan_known_360
✓  strict_lte_loose_rowwise
✓  prevalence_advisory


## 5. Zone-level prevalence

In [6]:

zone_df = label_by_zone(pass_instances, "dangerous_progression_k")
zone_df.pivot(index="zone_y", columns="zone_x", values="prevalence")


zone_x,0,1,2,3,4,5,6,7,8,9,10,11
zone_y,,,,,,,,,,,,
0,0.500000,0.407895,0.335227,0.261044,0.181818,0.178462,0.271284,0.455679,0.641860,0.848178,0.918605,0.827160
1,0.432432,0.442308,0.302368,0.231214,0.187648,0.207711,0.342756,0.564792,0.828369,0.949904,0.911111,0.820690
2,0.365482,0.340659,0.320280,0.216738,0.191262,0.220513,0.435205,0.671105,0.886824,0.971111,0.960000,0.849711
3,0.283550,0.265997,0.335835,0.255952,0.215205,0.274466,0.436700,0.658135,0.912162,0.973881,0.972222,0.894737
4,0.247892,0.258575,0.306195,0.247887,0.212555,0.286029,0.446053,0.723014,0.906863,0.963415,0.945055,0.777778
5,0.368159,0.377778,0.293967,0.213892,0.179941,0.223077,0.444584,0.672258,0.882353,0.977324,0.958333,0.838028
6,0.572650,0.419608,0.322330,0.214286,0.162416,0.214198,0.343303,0.594286,0.863354,0.945946,0.921296,0.851190
7,0.537037,0.410811,0.363636,0.254335,0.182515,0.183942,0.276151,0.467041,0.683735,0.877193,0.902834,0.731092
